In [ ]:
import sys
from pathlib import Path

sys.path.append("..")

import torch
from lightning import seed_everything
from torch_geometric.loader import NeighborLoader

from src import Module
from src.constants import DEFAULT_SEED
from src.graph import KNNGraph
from src.transforms import LineGraph

In [ ]:
BATCH_SIZE = 16
CKPT = "../lightning_logs/version_45927073/checkpoints/epoch=101-step=131378.ckpt"

In [ ]:
_ = seed_everything(DEFAULT_SEED, verbose=False)

In [ ]:
ckpt = torch.load(CKPT, map_location="cpu", weights_only=False)
k = ckpt["datamodule_hyper_parameters"]["k"]
ckpt_params = ckpt["hyper_parameters"] | ckpt["datamodule_hyper_parameters"]
k

In [ ]:
from ase.io import read

structure = Path().absolute().parent / "data" / "test" / "raw" / "POSCAR_CUBIC"
atoms = read(structure)

In [ ]:
knn = KNNGraph(k=k)

data = knn.convert(atoms)
lg_data = LineGraph().forward(data)

num_nodes = lg_data.num_nodes
if num_nodes is None:
    raise ValueError("The number of nodes in the graph is undefined.")

In [ ]:
model = Module.load_from_checkpoint(CKPT, weights_only=False, **ckpt_params)
model = model.eval()

In [ ]:
num_layers = ckpt_params["model_kwargs"]["n_bond_conv"]
loader = NeighborLoader(
    lg_data,
    num_neighbors=[k] * num_layers,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

In [ ]:
with torch.inference_mode():
    # for batch in loader:
    out = model(lg_data)

In [ ]:
preds = out.argmax(dim=1)

In [ ]:
lg_data.bond_source.unique(return_counts=True)

In [ ]:
lg_data.bond_target.unique(return_counts=True)